In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import json
from typing import Dict, Any

# 将项目根目录添加到 Python 路径，以便导入项目模块
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    print(f"Adding project root to Python path: {project_root}")
    sys.path.insert(0, project_root)

# 导入生成器
from implement.generator import MTHFVRPGenerator

In [ ]:
from implement.generator import VARIANT_GENERATION_PRESETS

print("所有支持的变体预设 (Keys):")
for key, value in VARIANT_GENERATION_PRESETS.items():
    print(f"{key}: {value}")

In [ ]:
def convert_problem_to_dict(problem: Dict[str, torch.Tensor], batch_idx: int) -> Dict[str, Any]:
    """
    将 TensorDict 中的单个问题实例转换为 Python 字典格式，
    移除 Tensor 包装并转为 list 或 scalar。
    """
    single_problem = {}
    for key, value in problem.items():
        if isinstance(value, torch.Tensor):
            # 处理不同维度的 Tensor
            if value.dim() == 0:
                single_problem[key] = value.item()
            elif value.dim() == 1: 
                single_problem[key] = value[batch_idx].item()
            else: # dim >= 2
                single_problem[key] = value[batch_idx].cpu().numpy().tolist()
        else:
            single_problem[key] = value
            
    # 构建符合数据规范的目标字典
    # 注意：这里会根据 generator 实际输出的字段进行提取
    converted_dict = {
        "locs": single_problem.get("locs"),
        "demand_backhaul": single_problem.get("demand_backhaul"), 
        "demand_linehaul": single_problem.get("demand_linehaul"),
        "backhaul_class": single_problem.get("backhaul_class"),
        "distance_limit": single_problem.get("distance_limit"),
        "time_windows": single_problem.get("time_windows"),
        "service_time": single_problem.get("service_time"),
        "available_vehicles": single_problem.get("available_vehicles"),
        "vehicle_capacity": single_problem.get("vehicle_capacity"),
        "vehicle_fixed_cost": single_problem.get("vehicle_fixed_cost"),
        "variable_cost": single_problem.get("variable_cost"), # 注意名称可能略有不同，视generator输出而定
        "open_route": single_problem.get("open_route"),
        "speed": single_problem.get("speed"),
        "heterogeneous_fleet": single_problem.get("heterogeneous_fleet")
    }
    
    return converted_dict

def save_problems_to_jsonl(problem_batch, filename: str):
    """
    将批量生成的问题保存到 JSONL 文件。
    """
    # 确保目录存在
    directory = os.path.dirname(filename)
    if directory and not os.path.exists(directory):
        os.makedirs(directory, exist_ok=True)
        print(f"创建目录: {directory}")
    
    # 获取批量大小
    batch_size = problem_batch.batch_size[0] if hasattr(problem_batch, 'batch_size') else len(problem_batch['locs'])
    
    with open(filename, 'w', encoding='utf-8') as f:
        for i in range(batch_size):
            # 转换单个问题
            problem_dict = convert_problem_to_dict(problem_batch, i)
            # 写入文件
            f.write(json.dumps(problem_dict, ensure_ascii=False) + '\n')
    
    print(f"成功保存 {batch_size} 个实例到: {filename}")

In [ ]:
import tqdm

# 生成配置
variants_to_generate = ["cvrp", "vrptw", "hfcvrp"]
nums_customers = 50   # 客户节点数 (不含 Depot)
num_veh = 10          # 车辆数
nums_problems = 1000   # 每个变体生成的实例数
output_dir = "../data" # 输出目录

for variant in tqdm.tqdm(variants_to_generate, desc="Generating Variants"):
    # 1. 初始化生成器
    # 注意：固定 random_seed 以确保结果可复现
    generator = MTHFVRPGenerator(
        num_loc=nums_customers, 
        variant_preset=variant, 
        vehicle_num=num_veh, 
        random_seed=42
    ) 

    # 2. 生成数据
    problem = generator(nums_problems)
    
    # 3. 生成问题签名 (Signature)    
    file_prefix = f"{variant}_node{nums_customers}_instance{nums_problems}"
    
    first_loc_start = problem['locs'][0][0].tolist()
    first_loc_end = problem['locs'][-1][0].tolist()
    
    sig = f"Signature [{file_prefix}]: {first_loc_start} -> {first_loc_end}"
    print(sig)

    # 4. 保存到文件
    save_path = os.path.join(output_dir, variant, f"{file_prefix}.jsonl")
    save_problems_to_jsonl(problem, save_path)